# Silver Layer: ERP Customer Location CDC (Free Edition)

**Pattern**: CDC via `foreachBatch` Merge  
**Trigger**: AvailableNow (Incremental Batch)

In [0]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable

bronze_table = "workspace.bronze.erp_loc_a101_cdc"
silver_table = "workspace.silver.erp_loc_cdc"
checkpoint_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/silver_erp_loc"

def upsert_erp_loc(batch_df, batch_id):
    transformed_df = batch_df.select(
        col("CID").alias("customer_number"),
        col("CNTRY").alias("country"),
        col("ingestion_timestamp")
    ).filter(col("customer_number").isNotNull())

    if not spark.catalog.tableExists(silver_table):
        transformed_df.write.format("delta").saveAsTable(silver_table)
        return

    target_table = DeltaTable.forName(spark, silver_table)
    (target_table.alias("t")
        .merge(transformed_df.alias("s"), "t.customer_number = s.customer_number")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

print(f"Starting incremental update for {silver_table}...")
query = (
    spark.readStream
    .table(bronze_table)
    .writeStream
    .foreachBatch(upsert_erp_loc)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
print(f"✓ Silver update complete. Total records in {silver_table}: {spark.table(silver_table).count():,}")